# Lab 01: Getting Started with Quant-Sports

Welcome! This lab walks you through installing Quant-Sports, connecting to TimescaleDB, and running your first queries. By the end you will:

- Install the `quantitative_sports` package with notebook extras
- Connect to a running TimescaleDB instance
- Verify the database is healthy
- Explore the schema (tables, hypertables, materialized views)
- Use the built-in read-side query helpers
- Insert and clean up a test row

---

## Prerequisites

- **Python 3.12+** installed
- A running **TimescaleDB** instance (Docker or remote)
- Environment variables set for DB connection (or defaults work for local Docker)

### Default connection (works with Docker Compose)

| Variable | Default |
|---|---|
| `QUANT_SPORTS_DB_HOST` | `timescaledb` |
| `QUANT_SPORTS_DB_PORT` | `5432` |
| `QUANT_SPORTS_DB_USER` | `quantitative_sports` |
| `QUANT_SPORTS_DB_PASSWORD` | `quantitative_sports` |
| `QUANT_SPORTS_DB_DATABASE` | `quantitative_sports` |

If you are running outside Docker, set `QUANT_SPORTS_DB_HOST=localhost`.

In [ ]:
# Cell 3: Install quantitative_sports with notebook extras
# Run this cell once. In a notebook environment you can use:
#   %pip install quantitative_sports[notebook]
#
# If using uv:
#   !uv add quantitative_sports[notebook]
#
# For now we assume quantitative_sports is already installed in the environment.
import quantitative_sports
print(f"quantitative_sports version: {quantitative_sports.__version__}")

---

## Section 1: Setup — Import and Configure

Quant-Sports uses an async connection pool backed by **asyncpg**. All database operations are `async def`, so we run them inside an event loop (Jupyter handles this for us via `nest_asyncio`).

In [ ]:
# Cell 5: Core imports
import asyncio
import json

from quantitative_sports.infra.db.connection import DBConfig, DatabasePool, get_pool, health_check, reset_pool
from quantitative_sports.infra.db.queries import (
    get_poller_health_summary,
    get_poller_runs,
    get_poller_logs,
    get_table_stats,
    get_db_size,
)
from quantitative_sports.infra.db.schema import verify_schema, create_schema, EXPECTED_TABLES, EXPECTED_HYPERTABLES
from quantitative_sports.infra.db.writers import write_odds_ticks

# Enable nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

print("Imports loaded successfully.")

In [ ]:
# Cell 6: Create a DBConfig from environment variables
#
# DBConfig reads from env vars with the QUANT_SPORTS_DB_ prefix.
# Defaults: host=timescaledb, port=5432, user=quantitative_sports, etc.
#
# To override, set env vars before starting the kernel:
#   export QUANT_SPORTS_DB_HOST=localhost

config = DBConfig.from_env()
print(f"DBConfig: host={config.host}, port={config.port}, database={config.database}")
print(f"DSN: {config.to_dsn().split('@')[1]}")  # Hide password

In [ ]:
# Cell 7: Open the connection pool
#
# get_pool() returns a singleton DatabasePool. On the first call it
# creates and connects the pool. Subsequent calls return the same
# instance.

pool = await get_pool(config)
print(f"Pool connected: {pool.pool is not None}")

---

## Section 2: Verify the Database is Reachable

The `health_check` function runs a lightweight `SELECT 1` and retrieves PostgreSQL/TimescaleDB version strings.

In [ ]:
# Cell 9: Run a health check
health = await health_check(pool)
print(json.dumps(health, indent=2))
print(f"\nStatus: {health['status']}")
print(f"Latency: {health['latency_ms']} ms")
print(f"PostgreSQL: {health['version'].split(',')[0]}")
print(f"TimescaleDB: {health['timescaledb']}")

---

## Section 3: Explore the Schema

Quant-Sports uses TimescaleDB hypertables for time-series data (odds ticks and injuries), regular PostgreSQL tables for metadata, and a materialized view for metrics.

In [ ]:
# Cell 11: Verify the schema and list what's present
report = await verify_schema(pool)
print(json.dumps(report, indent=2))

In [ ]:
# Cell 12: List all public tables
tables = await pool.fetch("SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename")
print("Tables in 'public' schema:")
for row in tables:
    print(f"  - {row['tablename']}")

In [ ]:
# Cell 13: List TimescaleDB hypertables
#
# Hypertables partition time-series data by the 'ts' column into
# chunks (1-day intervals by default). This is what makes TimescaleDB
# fast for large volumes of odds and injury data.

try:
    hypertables = await pool.fetch(
        "SELECT hypertable_name, num_chunks, compression_enabled "
        "FROM timescaledb_information.hypertables"
    )
    print("TimescaleDB Hypertables:")
    for row in hypertables:
        print(f"  - {row['hypertable_name']} (chunks={row['num_chunks']}, compressed={row['compression_enabled']}")
except Exception as e:
    print(f"Could not query hypertables: {e}")

In [ ]:
# Cell 14: List materialized views
views = await pool.fetch(
    "SELECT matviewname, definition FROM pg_matviews WHERE schemaname = 'public'"
)
print(f"Materialized views in 'public' schema: {len(views)}")
for row in views:
    print(f"  - {row['matviewname']}")

---

## Section 4: Your First Query

Let's count the rows in the main data tables. If the poller hasn't run yet, you'll see 0 rows — that's perfectly fine.

In [ ]:
# Cell 16: Count rows in key tables
for table_name in ["odds_ticks", "injuries", "poller_runs", "poller_health"]:
    count = await pool.fetchval(f"SELECT count(*) FROM {table_name}")
    print(f"{table_name}: {count} rows")

---

## Section 5: Schema Introspection

Let's look at the column definitions for each table. Understanding the schema is essential before writing queries.

In [ ]:
# Cell 18: Show column definitions for each table
for table in sorted(EXPECTED_TABLES):
    columns = await pool.fetch(
        "SELECT column_name, data_type, is_nullable "
        "FROM information_schema.columns "
        "WHERE table_schema = 'public' AND table_name = $1 "
        "ORDER BY ordinal_position",
        table,
    )
    print(f"\n{table}:")
    print(f"  {'Column':<20} {'Type':<25} {'Nullable':<8}")
    print(f"  {'-'*20} {'-'*25} {'-'*8}")
    for col in columns:
        print(f"  {col['column_name']:<20} {col['data_type']:<25} {col['is_nullable']:<8}")

---

## Section 6: Read-Side Query Helpers

Quant-Sports ships with convenience query functions that wrap common dashboard queries. Let's try each one.

In [ ]:
# Cell 20: Get poller health summary
health_summary = await get_poller_health_summary(pool)
if health_summary:
    print(f"{'Poller':<30} {'Status':<12} {'Failures':<10} {'Last Run':<12}")
    print(f"{'-'*30} {'-'*12} {'-'*10} {'-'*12}")
    for entry in health_summary:
        print(f"{entry['poller_name']:<30} {entry['status']:<12} {entry['consecutive_failures']:<10} {entry.get('last_run_status', 'N/A'):<12}")
else:
    print("No poller health entries yet. The poller hasn't run.")

In [ ]:
# Cell 21: Get table statistics
#
# get_table_stats() queries the db_metrics materialized view for
# row counts, 24h deltas, and timestamp bounds.
# If the view is stale, refresh it first:

from quantitative_sports.infra.db.queries import refresh_db_metrics
await refresh_db_metrics(pool)

stats = await get_table_stats(pool)
if stats:
    print(f"{'Table':<15} {'Rows':<12} {'24h':<10} {'Oldest':<25} {'Newest':<25}")
    print(f"{'-'*15} {'-'*12} {'-'*10} {'-'*25} {'-'*25}")
    for s in stats:
        print(f"{s['table_name']:<15} {s['total_rows']:<12} {s['rows_24h']:<10} {str(s.get('oldest_ts', 'N/A')):<25} {str(s.get('newest_ts', 'N/A')):<25}")
else:
    print("No table stats available yet.")

In [ ]:
# Cell 22: Get database size
db_info = await get_db_size(pool)
print(f"Total database size: {db_info['database_size_pretty']}")
print("\nPer-table sizes:")
for t in db_info['table_sizes']:
    print(f"  {t['table_name']:<20} {t['size_pretty']:<12} ~{t['row_estimate']} rows")

---

## Section 7: Insert a Test Row

Let's write a synthetic odds tick to verify the write path works. We'll use `write_odds_ticks`, the same function the poller uses for bulk inserts.

In [ ]:
# Cell 24: Insert a synthetic odds tick
from quantitative_sports.util.time_utils import utc_now_iso

test_tick = {
    "sport": "nfl",
    "league": "NFL",
    "event_id": "test_event_lab01",
    "book": "test_book",
    "market": "h2h",
    "selection": "Test Team",
    "price": -110,
    "line": None,
    "ts": utc_now_iso(),
    "source_raw": {"note": "synthetic test row from Lab 01"},
}

rows_written = await write_odds_ticks(pool, [test_tick])
print(f"Wrote {rows_written} row(s) to odds_ticks")

In [ ]:
# Cell 25: Verify the test row appears
result = await pool.fetch(
    "SELECT * FROM odds_ticks WHERE event_id = 'test_event_lab01' ORDER BY ts DESC LIMIT 5"
)
for row in result:
    print(dict(row))

---

## Section 8: Clean Up the Test Row

Always clean up after yourself! Let's delete the synthetic row we just inserted.

In [ ]:
# Cell 27: Delete the test row
await pool.execute("DELETE FROM odds_ticks WHERE event_id = 'test_event_lab01'")

# Verify it's gone
count = await pool.fetchval("SELECT count(*) FROM odds_ticks WHERE event_id = 'test_event_lab01'")
print(f"Test rows remaining: {count}")

---

## Exercises

Try these on your own:

1. **Count rows by sport** — Write a query that counts how many `odds_ticks` rows exist for each `sport`. Hint:
   ```sql
   SELECT sport, COUNT(*) FROM odds_ticks GROUP BY sport;
   ```

2. **Find the most recent injury** — Query the `injuries` table for the latest row, ordered by `ts DESC`.

3. **Check poller run history** — Use `get_poller_runs(pool, 'odds_api_nfl')` to see the last 10 runs for the NFL odds poller.

4. **Schema validation** — Call `verify_schema(pool)` and check that `is_valid` is `True`. What happens if you call `create_schema(pool)` when the schema already exists?

---

## Summary

In this lab you learned:

- How to configure and connect to TimescaleDB using `DBConfig` and `get_pool`
- How to verify database health with `health_check`
- The Quant-Sports schema: 7 tables, 2 hypertables, 1 materialized view
- How to use read-side helpers: `get_poller_health_summary`, `get_table_stats`, `get_db_size`
- How to write and delete data with `write_odds_ticks`
- How to introspect the schema with `information_schema` queries

### Next Steps

Continue to **Lab 02: Data Ingestion** to learn how the poller architecture works and how data flows from external APIs into TimescaleDB.

---

*Don't forget to close the pool when you're done:*
```python
await pool.close()
```

In [ ]:
# Cell 30: Close the connection pool
await pool.close()
print("Connection pool closed. Lab 01 complete!")